# F1 Race Strategy Simulator - exploration

## Part 1: loading a real race

Uses `src/data_loader.py` to load the 2023 Bahrain Grand Prix and inspect the cleaned lap table.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Make ``import src...`` work wherever Jupyter was started from
# (this notebook lives in notebooks/, the package in the project root).
root = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()
)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.data_loader import (  # noqa: E402
    get_lap_data, get_race_list, load_race_session, setup_cache,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [ ]:
# Enable the cache first (creates <project>/data if it does not exist).
print("Cache folder:", setup_cache())

# Completed 2023 races - pick one by EventName or RoundNumber.
print(get_race_list(2023).to_string(index=False))

In [ ]:
# The first run downloads the data; later runs read it from data/.
session = load_race_session(2023, "Bahrain", "R")
laps = get_lap_data(session)

print(
    f"{session.event['EventName']} {session.event.year}: "
    f"{len(laps)} laps, {laps['Driver'].nunique()} drivers"
)
print(laps.head(10))

In [ ]:
print("Compounds used:", sorted(laps["Compound"].unique()))

stints = laps.groupby("Driver")["Stint"].nunique().rename("Stints").to_frame()
stints["Stops"] = stints["Stints"] - 1
print("\nStints per driver:")
print(stints.sort_values("Stints", ascending=False))

print("\nFlagged laps (kept in the table, not dropped):")
print(laps[["MissingLapTime", "IsInLap", "IsOutLap"]].sum().to_string())

print("\nTrackStatus values seen (lap counts):")
print(laps["TrackStatus"].value_counts().to_string())